# Vesper Capital — Wallet Search POC v2

Two-stage targeted search for skilled ETH wallets.

**Stage 1 — Candidate generation** (~$0.003 at default settings)
Pull large ETH transfer events from the last 90 days. Apply cheap behavioral filters to reduce 500k+ active wallets to a high-quality candidate pool without any balance history fetches.

**Stage 2 — Sequential SPRT scoring** (~$0.006-0.037/wallet)
Fetch balance history in 30-day increments using the fast async `fetch_balances_for_schedule` from core utils. Apply Sequential Probability Ratio Test after each increment: accept (skilled), abandon (not skilled), or continue. Most wallets stop after 1-2 increments — dramatically reducing cost vs full history fetch.

**Network expansion** (~$0.10)
Find frequent counterparties of confirmed skilled wallets. Sophisticated participants transact with each other.

**Changes from v1:**
- Uses `fetch_balances_for_schedule` (async batching) instead of sequential HTTP requests — 100x faster per wallet
- Block map auto-extended to cover current dates
- NaN-safe weighted edge computation throughout
- Max mean trade size filter to exclude exchange/whale wallets
- Trade returns/weights collected with proper NaN filtering
- Network expansion runs before Stage 2, prioritizing best leads

**Estimated total cost: $3-6 for a full run**

## 1. Configuration & cost estimate

In [1]:
from claude_refactor_core_utils import *
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import pickle
from datetime import datetime, date, timedelta
from pathlib import Path
from collections import defaultdict
from scipy import stats as scipy_stats

ALCHEMY_API_KEY   = "8J0Ou3_XsuWVbOiZ0xpEvioGx7k0wab9"
ETHERSCAN_API_KEY = "GH6FVBANEIR1HS6PEN36XMYQRY2IDEAAXR"
BASE_URL          = f"https://eth-mainnet.g.alchemy.com/v2/{ALCHEMY_API_KEY}"

# ================================================================
# BUDGET CONTROLS — adjust before running
# ================================================================

# Stage 1: candidate generation from event logs
STAGE1_MAX_LOG_CALLS     = 100      # ~$0.003 at 75 CUs each
STAGE1_LOOKBACK_DAYS     = 90       # how far back to scan
STAGE1_MIN_TRADE_SIZE    = 0.1      # min ETH per trade
STAGE1_MAX_MEAN_TRADE    = 50.0     # max mean ETH — exclude exchanges/whales
STAGE1_MIN_TRADE_COUNT   = 5        # min trades in lookback
STAGE1_MAX_WALLETS       = 10_000   # max candidates forwarded to Stage 2

# Stage 2: sequential scoring
STAGE2_MAX_CANDIDATES    = 50       # wallets to score (increase for full run)
STAGE2_SCORING_DAYS      = 30       # days per scoring increment
STAGE2_MAX_INCREMENTS    = 6        # max increments = 180 days max history
STAGE2_SPRT_LOWER        = 0.1      # abandon if LR < this
STAGE2_SPRT_UPPER        = 10.0     # accept if LR > this
STAGE2_MIN_TRADES        = 10       # min trades before SPRT decides
MAX_CU_THIS_RUN          = 5_000_000  # hard cap per run (~$2.25)

# Known exchange / MEV addresses to exclude
MEV_CONTRACTS = {
    '0x00000000003b3cc22af3ae1eac0440bcee416b40',
    '0x00000000219ab540356cbb839cbe05303d7705fa',
    '0x3f5ce5fbfe3e9af3971dd833d26ba9b5c936f0be',
    '0x28c6c06298d514db089934071355e5743bf21d60',  # Binance hot wallet
    '0x21a31ee1afc51d94c2efccaa2092ad1028285549',  # Binance
    '0xdfd5293d8e347dfe59e90efd55b2956a1343963d',  # Binance
}

# Paths
BASE_DIR   = Path('/home/colin/projects/blockchain_alpha')
SEARCH_DIR = BASE_DIR / 'data/wallet_search'
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_CSV   = SEARCH_DIR / 'stage1_candidates.csv'
STAGE2_CSV   = SEARCH_DIR / 'stage2_scored.csv'
ACCEPTED_CSV = SEARCH_DIR / 'accepted_wallets.csv'
STATE_PKL    = SEARCH_DIR / 'search_state.pkl'

# CU/cost constants
cu_per_log_call     = 75
cu_per_balance      = 19
cu_rate_per_million = 0.45

# Cost estimates
s1_est_cu  = STAGE1_MAX_LOG_CALLS * cu_per_log_call
s2_est_cu  = STAGE2_MAX_CANDIDATES * STAGE2_MAX_INCREMENTS * STAGE2_SCORING_DAYS * 24 * cu_per_balance
tot_est_cu = s1_est_cu + s2_est_cu

print("=" * 60)
print("VESPER CAPITAL — WALLET SEARCH POC v2")
print("=" * 60)
print()
print(f"Stage 1  est. cost: ${s1_est_cu/1e6*cu_rate_per_million:.3f}")
print(f"Stage 2  est. cost (worst case): ${s2_est_cu/1e6*cu_rate_per_million:.2f}")
print(f"Total    est. cost: ${tot_est_cu/1e6*cu_rate_per_million:.2f}")
print()
print("Adjust STAGE1_MAX_LOG_CALLS and STAGE2_MAX_CANDIDATES to control spend.")


VESPER CAPITAL — WALLET SEARCH POC v2

Stage 1  est. cost: $0.003
Stage 2  est. cost (worst case): $1.85
Total    est. cost: $1.85

Adjust STAGE1_MAX_LOG_CALLS and STAGE2_MAX_CANDIDATES to control spend.


## 2. Load & extend block map

Loads the existing hourly block map and extends it forward to today using block interpolation (~300 blocks/hour). Safe to run multiple times.

In [2]:
# ================================================================
# Load and extend block map to cover current scoring window
# ================================================================

print("Loading block map...")
bm_path = BASE_DIR / "data/hourly_block_map.csv"
if bm_path.exists():
    bm_df = pd.read_csv(bm_path)
    bm_df["timestamp"] = pd.to_datetime(bm_df["timestamp"]).dt.floor("h")
    block_map = dict(zip(bm_df["timestamp"], bm_df["block_number"].astype(int)))
    print(f"Loaded: {len(block_map):,} entries")
    print(f"Range:  {min(block_map)} to {max(block_map)}")
else:
    block_map = {}
    print("No block map found — will build from scratch")

# Extend forward to today if needed
def get_current_block():
    resp = requests.post(BASE_URL, json={
        "jsonrpc": "2.0", "method": "eth_blockNumber", "params": [], "id": 1
    }, timeout=10)
    return int(resp.json()["result"], 16)

current_block = get_current_block()
current_time  = pd.Timestamp.utcnow().floor("h").tz_localize(None)

if block_map:
    last_known_time  = max(block_map.keys())
    last_known_block = block_map[last_known_time]
    hours_gap        = int((current_time - last_known_time).total_seconds() / 3600)
    blocks_gap       = current_block - last_known_block
    blocks_per_h     = blocks_gap / hours_gap if hours_gap > 0 else 300

    if hours_gap > 0:
        print(f"Extending block map forward {hours_gap:,} hours...")
        for h in range(1, hours_gap + 1):
            ts = last_known_time + pd.Timedelta(hours=h)
            block_map[ts] = int(last_known_block + blocks_per_h * h)

        # Save updated map
        pd.DataFrame({
            "timestamp":    list(block_map.keys()),
            "block_number": list(block_map.values())
        }).sort_values("timestamp").to_csv(bm_path, index=False)
        print(f"Block map saved. New range: {min(block_map)} to {max(block_map)}")
else:
    # Bootstrap from current block
    print("Building block map from current block (~300 blocks/hr)...")
    blocks_per_h = 300
    for h in range(24 * 200):  # 200 days back
        ts = current_time - pd.Timedelta(hours=h)
        block_map[ts] = max(1, current_block - blocks_per_h * h)
    print(f"Built {len(block_map):,} entries")

print(f"Current block: {current_block:,}")


Loading block map...
Loaded: 29,307 entries
Range:  2023-01-01 00:00:00 to 2026-05-06 02:00:00
Extending block map forward 1 hours...
Block map saved. New range: 2023-01-01 00:00:00 to 2026-05-06 03:00:00
Current block: 25,033,236


## 3. Load price data for scoring window

In [3]:
# ================================================================
# Load price data for scoring window
# ================================================================

SCORING_END   = date.today() - timedelta(days=1)
SCORING_START = SCORING_END - timedelta(
    days=STAGE2_MAX_INCREMENTS * STAGE2_SCORING_DAYS + 30
)

print(f"Loading price data: {SCORING_START} to {SCORING_END}...")
df_price = load_eth_usd_cryptocompare(
    start=str(SCORING_START), end=str(SCORING_END), interval="1h"
)
df_price["date"] = pd.to_datetime(df_price["date"]).dt.floor("h")
df_price = df_price.sort_values("date").reset_index(drop=True)
df_price["fwd_return"] = (
    df_price["close"].shift(-24) - df_price["close"]
) / df_price["close"] * 100

BASELINE_RETURN = df_price["fwd_return"].dropna().mean()
SKILLED_RETURN  = BASELINE_RETURN + 0.5
RETURN_SIGMA    = df_price["fwd_return"].dropna().std()

print(f"Price data: {len(df_price):,} hours")
print(f"Baseline 24h return: {BASELINE_RETURN:+.4f}%")
print(f"Skilled threshold:   {SKILLED_RETURN:+.4f}%")
print(f"Return std dev:      {RETURN_SIGMA:.4f}%")


Loading price data: 2025-10-06 to 2026-05-04...
Price data: 5,041 hours
Baseline 24h return: -0.2652%
Skilled threshold:   +0.2348%
Return std dev:      3.5423%


## 4. Stage 1 — Candidate generation

Pulls large ETH transfer events and scores candidates by behavioral patterns. **Run cells 4 and 5 together.**

In [4]:
# ================================================================
# Stage 1 — Candidate generation from transfer event logs
#
# Filters (cheapest first):
#   1. Active in last STAGE1_LOOKBACK_DAYS days
#   2. Min trade frequency >= STAGE1_MIN_TRADE_COUNT
#   3. Min mean trade size >= STAGE1_MIN_TRADE_SIZE ETH
#   4. Max mean trade size <= STAGE1_MAX_MEAN_TRADE (exclude whales/exchanges)
#   5. Exclude known MEV / exchange addresses
#   6. Accumulator pattern score (consistent sizing)
# ================================================================

def fetch_large_eth_transfers(from_block, to_block, min_value_wei, max_calls=20):
    transfers  = []
    call_count = 0
    payload = {
        "jsonrpc": "2.0",
        "method":  "alchemy_getAssetTransfers",
        "params":  [{
            "fromBlock":      hex(from_block),
            "toBlock":        hex(to_block),
            "category":       ["external"],
            "withMetadata":   True,
            "excludeZeroValue": True,
            "maxCount":       "0x3e8",
            "order":          "asc",
        }],
        "id": 1
    }
    page_key = None
    while call_count < max_calls:
        if page_key:
            payload["params"][0]["pageKey"] = page_key
        resp = requests.post(BASE_URL, json=payload, timeout=30)
        data = resp.json().get("result", {})
        batch = data.get("transfers", [])
        call_count += 1
        for t in batch:
            val = float(t.get("value", 0) or 0)
            if val * 1e18 >= min_value_wei:
                transfers.append({
                    "from":      t.get("from", "").lower(),
                    "to":        t.get("to",   "").lower(),
                    "value_eth": val,
                    "block":     int(t.get("blockNum", "0x0"), 16),
                    "timestamp": t.get("metadata", {}).get("blockTimestamp", ""),
                })
        page_key = data.get("pageKey")
        if not page_key:
            break
    return transfers, call_count

est_start_block = current_block - (STAGE1_LOOKBACK_DAYS * 7200)
min_value_wei   = int(STAGE1_MIN_TRADE_SIZE * 1e18)

print(f"Stage 1: fetching transfers from block {est_start_block:,} to {current_block:,}")
print(f"Min trade size: {STAGE1_MIN_TRADE_SIZE} ETH  |  Max log calls: {STAGE1_MAX_LOG_CALLS}")
print()

all_transfers = []
total_calls   = 0
chunk_start   = est_start_block
CHUNK_SIZE    = 50_000

while chunk_start < current_block and total_calls < STAGE1_MAX_LOG_CALLS:
    chunk_end = min(chunk_start + CHUNK_SIZE, current_block)
    transfers, calls = fetch_large_eth_transfers(
        chunk_start, chunk_end, min_value_wei,
        max_calls=min(20, STAGE1_MAX_LOG_CALLS - total_calls)
    )
    all_transfers.extend(transfers)
    total_calls   += calls
    chunk_start    = chunk_end + 1
    print(f"  [{total_calls}/{STAGE1_MAX_LOG_CALLS} calls] "
          f"block {chunk_start:,}  transfers: {len(all_transfers):,}")
    if total_calls >= STAGE1_MAX_LOG_CALLS:
        break

print(f"\nFetched {len(all_transfers):,} transfers using {total_calls} calls")
print(f"Stage 1 cost: ${total_calls * cu_per_log_call / 1e6 * cu_rate_per_million:.4f}")


Stage 1: fetching transfers from block 24,385,236 to 25,033,236
Min trade size: 0.1 ETH  |  Max log calls: 100

  [20/100 calls] block 24,435,237  transfers: 4,692
  [40/100 calls] block 24,485,238  transfers: 7,584
  [60/100 calls] block 24,535,239  transfers: 10,001
  [80/100 calls] block 24,585,240  transfers: 14,747
  [100/100 calls] block 24,635,241  transfers: 19,065

Fetched 19,065 transfers using 100 calls
Stage 1 cost: $0.0034


In [5]:
# ================================================================
# Stage 1 filtering and candidate scoring
# ================================================================

if not all_transfers:
    print("No transfers fetched.")
    candidates = pd.DataFrame()
else:
    df_t = pd.DataFrame(all_transfers)
    df_t["timestamp"] = pd.to_datetime(df_t["timestamp"], format="mixed", errors="coerce")
    df_t = df_t.dropna(subset=["timestamp"]).reset_index(drop=True)

    # Exclude known bad addresses
    df_t = df_t[~df_t["from"].isin(MEV_CONTRACTS)]
    df_t = df_t[~df_t["to"].isin(MEV_CONTRACTS)]

    # Focus on receiving wallets (accumulation)
    buyers = df_t.groupby("to").agg(
        n_buys       = ("value_eth", "count"),
        total_eth_in = ("value_eth", "sum"),
        mean_eth_in  = ("value_eth", "mean"),
        std_eth_in   = ("value_eth", "std"),
        first_seen   = ("timestamp", "min"),
        last_seen    = ("timestamp", "max"),
    ).reset_index().rename(columns={"to": "address"})

    # Trade count and size filters
    buyers = buyers[
        (buyers["n_buys"]     >= STAGE1_MIN_TRADE_COUNT) &
        (buyers["mean_eth_in"] >= STAGE1_MIN_TRADE_SIZE)  &
        (buyers["mean_eth_in"] <= STAGE1_MAX_MEAN_TRADE)    # exclude whales
    ]

    # Valid address length only
    buyers = buyers[buyers["address"].str.len() == 42]

    # Accumulator score: consistent sizing (low CV = more deliberate)
    buyers["trade_cv"] = (
        buyers["std_eth_in"] / buyers["mean_eth_in"].clip(lower=0.001)
    ).fillna(999).clip(upper=10)

    # Composite candidate score
    buyers["candidate_score"] = (
        buyers["n_buys"] *
        buyers["mean_eth_in"] /
        (buyers["trade_cv"] + 0.1)
    )
    buyers = buyers.sort_values("candidate_score", ascending=False)

    candidates = buyers.head(STAGE1_MAX_WALLETS).reset_index(drop=True)
    candidates.to_csv(STAGE1_CSV, index=False)

    print(f"Stage 1 complete:")
    print(f"  Raw transfers:       {len(df_t):,}")
    print(f"  Unique buyers:       {buyers.shape[0]:,}")
    print(f"  Top candidates:      {len(candidates):,}")
    print()
    print("Top 20 candidates:")
    print(candidates[[
        "address","n_buys","mean_eth_in","trade_cv","candidate_score"
    ]].head(20).round(3).to_string(index=False))


Stage 1 complete:
  Raw transfers:       18,226
  Unique buyers:       279
  Top candidates:      279

Top 20 candidates:
                                   address  n_buys  mean_eth_in  trade_cv  candidate_score
0x4097dcf525f58acfc08952f184f3900961fa43e9      36        7.291     0.000         2624.589
0xa1abfa21f80ecf401bd41365adbb6fef6fefdf09     131       43.939     3.438         1627.048
0x44894aeee56c2dd589c1d5c8cb04b87576967f97       5       32.000     0.000         1600.000
0xb5d85cbf7cb3ee0d56b3bb207d5fc4b82f43f511     254       16.664     5.041          823.267
0xa9ac43f5b5e38155a288d1a01d2cbc4478e14573      91       32.032     3.672          772.769
0xc36442b4a4522e871399cd717abdd847ab11fe88      39       32.852     1.951          624.744
0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb       5       33.476     0.184          589.555
0x5b702b24b00a37b61ff9d71d91fed1a629edd1df       6       39.833     0.386          491.972
0xf30ba13e4b04ce5dc4d254ae5fa95477800f0eb0     200       12

## 5. Network expansion

Find counterparties of confirmed skilled wallets. These are the highest-quality Stage 2 candidates and are scored first.

In [6]:
# ================================================================
# Network expansion — find counterparties of known skilled wallets
# Run before Stage 2 to prioritize the best leads
# ================================================================

existing_skilled = [
    "0x1b4b91bb747176451f0f7f73b636babafcf31cc4",
]
if ACCEPTED_CSV.exists():
    existing_skilled += pd.read_csv(ACCEPTED_CSV)["address"].tolist()

print(f"Network expansion from {len(existing_skilled)} seed wallets...")

def get_counterparties(address, alchemy_api_key, max_transfers=200):
    counterparties = defaultdict(float)
    for direction in ["fromAddress", "toAddress"]:
        payload = {
            "jsonrpc": "2.0",
            "method":  "alchemy_getAssetTransfers",
            "params":  [{
                direction:      address,
                "category":     ["external"],
                "withMetadata": True,
                "maxCount":     hex(max_transfers),
                "order":        "desc",
            }],
            "id": 1
        }
        resp = requests.post(
            f"https://eth-mainnet.g.alchemy.com/v2/{alchemy_api_key}",
            json=payload, timeout=15
        )
        for t in resp.json().get("result", {}).get("transfers", []):
            other = t.get(
                "to" if direction == "fromAddress" else "from", ""
            ).lower()
            val = float(t.get("value", 0) or 0)
            if other and other not in MEV_CONTRACTS and len(other) == 42:
                counterparties[other] += val
    return sorted(counterparties.items(), key=lambda x: x[1], reverse=True)

network_candidates = {}
for seed in existing_skilled:
    print(f"  Expanding from {seed[:20]}...")
    try:
        for addr, eth in get_counterparties(seed, ALCHEMY_API_KEY)[:20]:
            if addr not in existing_skilled:
                network_candidates[addr] = network_candidates.get(addr, 0) + eth
    except Exception as e:
        print(f"  Error: {e}")

network_df = pd.DataFrame([
    {"address": addr, "total_eth_with_seeds": eth}
    for addr, eth in sorted(
        network_candidates.items(), key=lambda x: x[1], reverse=True
    )
])

if len(network_df) > 0:
    network_path = SEARCH_DIR / "network_candidates.csv"
    network_df.to_csv(network_path, index=False)
    print(f"\nNetwork candidates: {len(network_df)}")
    print(network_df.head(20).to_string(index=False))
else:
    print("No network candidates found.")
    network_df = pd.DataFrame(columns=["address", "total_eth_with_seeds"])


Network expansion from 1 seed wallets...
  Expanding from 0x1b4b91bb747176451f...

Network candidates: 20
                                   address  total_eth_with_seeds
0xdfaa75323fb721e5f29d43859390f62cc4b600b8             27.491500
0x974caa59e49682cda0ad2bbe82983419a2ecc400             20.872838
0x2af4feb9a15088f4c4a185b1b74950c48483c1f1             16.360080
0x881d40237659c251811cec9c364ef91dc08d300c              5.340000
0x88ac52fb7a432fe1b87ba04f3782a91bd981cc77              4.645255
0x90e6d782895ad20b23a7f05169e0480bebbd9adf              4.500000
0x09cd7fc18ab2f8d698998ecc0b0de7afbec757a3              4.000000
0xaacbf5550deaca37cfd03479350c9cb702986d7a              3.730058
0x5cfc00d4e692b98c48729c7732bb3a68abb1519e              3.564624
0x83e327d80e38282c208a9dda02deb276d9edc6da              3.500000
0x307576dd4f73f91bb8c4a2edb762938e8e067d31              3.270123
0x192480da2e92beb0ba4d52e0e4709afe312ad186              1.985954
0x082d2397c5e3c5300d31f52bd056169882b8ba1c       

## 6. Stage 2 helper functions

Define `compute_skill_score`, `sprt_update`, and `safe_wtd_edge`. Run once before the Stage 2 loop.

In [7]:
# ================================================================
# Stage 2 helper functions
# ================================================================

def compute_skill_score(balance_df, price_df, target_hours=24, min_delta=0.005):
    """Size-weighted forward return skill score. Returns (edge, n_trades, p_val)."""
    bal = balance_df.copy()
    bal["delta_eth"] = bal["balance_eth"].diff()
    bal["action"]    = bal["delta_eth"].apply(
        lambda d: 1 if d > min_delta else (-1 if d < -min_delta else 0)
    )
    buys = bal[bal["action"] == 1].copy()
    if len(buys) < 3:
        return 0.0, 0, 1.0

    buys = buys.merge(
        price_df[["date","fwd_return"]].dropna(subset=["fwd_return"]),
        on="date", how="inner"
    )
    if len(buys) < 3:
        return 0.0, 0, 1.0

    baseline = price_df["fwd_return"].dropna().mean()
    weights  = np.nan_to_num(buys["delta_eth"].clip(lower=0).values, nan=0.0)

    if weights.sum() > 0:
        mean_edge = float(np.average(buys["fwd_return"], weights=weights) - baseline)
    else:
        mean_edge = float(buys["fwd_return"].mean() - baseline)

    if np.isnan(mean_edge):
        mean_edge = 0.0

    t_stat, p_val = scipy_stats.ttest_1samp(buys["fwd_return"].values, baseline)
    return mean_edge, len(buys), float(p_val)


def sprt_update(likelihood_ratio, observed_return, baseline_mean,
                skilled_mean, sigma, lower=0.1, upper=10.0):
    """Update SPRT likelihood ratio with one new observation."""
    from scipy.stats import norm
    p0 = norm.pdf(observed_return, baseline_mean, sigma)
    p1 = norm.pdf(observed_return, skilled_mean,  sigma)
    if p0 == 0:
        new_ratio = likelihood_ratio * 10
    else:
        new_ratio = likelihood_ratio * (p1 / p0)
    new_ratio = float(np.clip(new_ratio, 1e-6, 1e6))
    if new_ratio >= upper:
        return new_ratio, "accept"
    elif new_ratio <= lower:
        return new_ratio, "abandon"
    return new_ratio, "continue"


def safe_wtd_edge(all_returns, all_weights, baseline):
    """Compute size-weighted edge, robust to NaN and zero weights."""
    if not all_returns:
        return 0.0
    returns = np.array(all_returns)
    weights = np.nan_to_num(np.array(all_weights), nan=0.0).clip(min=0)
    valid   = ~np.isnan(returns)
    returns = returns[valid]
    weights = weights[valid]
    if len(returns) == 0:
        return 0.0
    if weights.sum() > 0:
        edge = float(np.average(returns, weights=weights) - baseline)
    else:
        edge = float(returns.mean() - baseline)
    return 0.0 if np.isnan(edge) else edge


print("Stage 2 helper functions defined.")
cu_per_inc = STAGE2_SCORING_DAYS * 24 * cu_per_balance
print(f"Cost per 30-day increment per wallet: "
      f"{cu_per_inc:,} CUs = ${cu_per_inc/1e6*cu_rate_per_million:.4f}")


Stage 2 helper functions defined.
Cost per 30-day increment per wallet: 13,680 CUs = $0.0062


## 7. Stage 2 — Sequential scoring

**Restart-safe** — saves state every 10 wallets. Re-run this cell to continue from where it stopped.

**Budget:** set `MAX_CU_THIS_RUN` in cell 1 to limit spend per run.

In [8]:
# ================================================================
# Stage 2 — Sequential scoring with SPRT early stopping
# Uses fetch_balances_for_schedule from core utils for speed
# Restart-safe: saves state every 10 wallets
# ================================================================

# Build candidate list: network candidates first, then Stage 1
network_path = SEARCH_DIR / "network_candidates.csv"
network_wallets = (pd.read_csv(network_path)["address"].tolist()
                   if network_path.exists() else [])

stage1_wallets = (pd.read_csv(STAGE1_CSV)["address"].tolist()
                  if STAGE1_CSV.exists() else [])

# Deduplicate, network candidates first
seen = set()
wallet_list = []
for addr in network_wallets + stage1_wallets:
    if addr not in seen:
        seen.add(addr)
        wallet_list.append(addr)
wallet_list = wallet_list[:STAGE2_MAX_CANDIDATES]

print(f"Candidates: {len(network_wallets)} network + {len(stage1_wallets)} stage1 "
      f"= {len(wallet_list)} total (capped at {STAGE2_MAX_CANDIDATES})")

# Warm start
if STATE_PKL.exists():
    with open(STATE_PKL, "rb") as f:
        state = pickle.load(f)
    print(f"Warm start: {len(state['results'])} already processed")
else:
    state = {
        "results":      {},
        "accepted":     [],
        "abandoned":    [],
        "in_progress":  {},
        "total_cu":     0,
    }

results     = state["results"]
accepted    = state["accepted"]
abandoned   = state["abandoned"]
in_progress = state["in_progress"]
total_cu    = state["total_cu"]
cu_this_run = 0

# Build block map DataFrame once for slicing
bm_df_full = pd.DataFrame({
    "timestamp":    list(block_map.keys()),
    "block_number": list(block_map.values())
}).sort_values("timestamp").reset_index(drop=True)

print(f"\nStarting Stage 2...")
print(f"  Budget this run: {MAX_CU_THIS_RUN:,} CUs (~${MAX_CU_THIS_RUN/1e6*cu_rate_per_million:.2f})")
print()

for wallet_idx, address in enumerate(wallet_list):

    if address in accepted or address in abandoned:
        continue
    if cu_this_run >= MAX_CU_THIS_RUN:
        print(f"\nBudget reached. Stopping at wallet {wallet_idx}/{len(wallet_list)}.")
        break

    if address not in in_progress:
        in_progress[address] = {
            "increment":        0,
            "likelihood_ratio": 1.0,
            "all_returns":      [],
            "all_weights":      [],
            "n_trades_total":   0,
        }

    ws       = in_progress[address]
    decision = "continue"

    print(f"Wallet {wallet_idx+1}/{len(wallet_list)}: {address[:20]}...  "
          f"[inc {ws['increment']+1}/{STAGE2_MAX_INCREMENTS}]", end="  ")

    while ws["increment"] < STAGE2_MAX_INCREMENTS and decision == "continue":
        if cu_this_run >= MAX_CU_THIS_RUN:
            break

        # Date window for this increment (working backwards from SCORING_END)
        inc_end   = SCORING_END - timedelta(days=STAGE2_SCORING_DAYS * ws["increment"])
        inc_start = inc_end    - timedelta(days=STAGE2_SCORING_DAYS)

        # Slice block map for this window
        window_bm = bm_df_full[
            (bm_df_full["timestamp"] >= pd.Timestamp(inc_start)) &
            (bm_df_full["timestamp"] <= pd.Timestamp(inc_end))
        ].reset_index(drop=True)

        if len(window_bm) == 0:
            ws["increment"] += 1
            continue

        # Build schedule
        schedule = pd.DataFrame({
            "address": address,
            "date":    pd.date_range(start=inc_start, end=inc_end, freq="h")
        })

        # Fetch balances using core utils async batching
        try:
            bal_raw = fetch_balances_for_schedule(
                schedule        = schedule,
                block_df        = window_bm.rename(columns={"timestamp":"timestamp"}),
                alchemy_api_key = ALCHEMY_API_KEY,
                batch_size      = 100,
                max_concurrency = 20,
                rps             = 25.0,
                progress_every  = 99999,
                block_date_col  = "timestamp",
                label           = f"{address[:10]}-i{ws['increment']+1}",
            )
        except Exception as e:
            print(f"[fetch error: {e}]", end=" ")
            ws["increment"] += 1
            continue

        cu_used       = len(schedule) * cu_per_balance
        cu_this_run  += cu_used
        total_cu     += cu_used

        if bal_raw is None or bal_raw.empty:
            ws["increment"] += 1
            continue

        # Parse balance data
        bal_raw["date"]        = pd.to_datetime(bal_raw["date"], format="mixed").dt.floor("h")
        bal_raw["balance_eth"] = (
            pd.to_numeric(bal_raw["balance_wei"], errors="coerce").fillna(0) / 1e18
        )
        bal_df = (bal_raw[["date","balance_eth"]]
                  .sort_values("date")
                  .reset_index(drop=True))

        # Compute per-increment skill score
        edge_inc, n_trades_inc, p_val = compute_skill_score(
            bal_df, df_price
        )
        ws["n_trades_total"] += n_trades_inc

        # Collect trade-level returns and weights for SPRT
        bal_df["delta_eth"] = bal_df["balance_eth"].diff()
        bal_df["action"]    = bal_df["delta_eth"].apply(
            lambda d: 1 if d > 0.005 else 0
        )
        buy_rows = bal_df[bal_df["action"] == 1][["date","delta_eth"]].copy()
        buy_rows = buy_rows.merge(
            df_price[["date","fwd_return"]].dropna(subset=["fwd_return"]),
            on="date", how="inner"
        )
        trade_returns = buy_rows["fwd_return"].tolist()
        trade_weights = buy_rows["delta_eth"].fillna(0.0).clip(lower=0).tolist()

        ws["all_returns"].extend(trade_returns)
        ws["all_weights"].extend(trade_weights)
        ws["increment"] += 1

        # Apply SPRT once we have enough trades
        if ws["n_trades_total"] >= STAGE2_MIN_TRADES:
            for ret in trade_returns:
                ws["likelihood_ratio"], decision = sprt_update(
                    ws["likelihood_ratio"], ret,
                    BASELINE_RETURN, SKILLED_RETURN, RETURN_SIGMA,
                    STAGE2_SPRT_LOWER, STAGE2_SPRT_UPPER
                )
                if decision != "continue":
                    break

    # Final decision when max increments reached without SPRT decision
    if decision == "continue" and ws["increment"] >= STAGE2_MAX_INCREMENTS:
        final_edge = safe_wtd_edge(
            ws["all_returns"], ws["all_weights"], BASELINE_RETURN
        )
        decision = "accept" if final_edge > 0.2 else "abandon"

    # Compute final edge
    wtd_edge = safe_wtd_edge(
        ws["all_returns"], ws["all_weights"], BASELINE_RETURN
    )

    results[address] = {
        "address":          address,
        "decision":         decision,
        "increments_used":  ws["increment"],
        "n_trades":         ws["n_trades_total"],
        "weighted_edge":    wtd_edge,
        "likelihood_ratio": ws["likelihood_ratio"],
        "cu_used":          ws["increment"] * STAGE2_SCORING_DAYS * 24 * cu_per_balance,
    }

    if decision == "accept":
        accepted.append(address)
        status = f"✓ ACCEPTED  edge={wtd_edge:+.4f}%  n={ws['n_trades_total']}  LR={ws['likelihood_ratio']:.2f}"
    elif decision == "abandon":
        abandoned.append(address)
        status = f"✗ abandoned edge={wtd_edge:+.4f}%  n={ws['n_trades_total']}  LR={ws['likelihood_ratio']:.2f}"
    else:
        status = f"~ in progress  edge={wtd_edge:+.4f}%  n={ws['n_trades_total']}"
    print(status)

    # Save state every 10 wallets
    if (wallet_idx + 1) % 10 == 0:
        state.update({
            "results": results, "accepted": accepted,
            "abandoned": abandoned, "in_progress": in_progress,
            "total_cu": total_cu,
        })
        with open(STATE_PKL, "wb") as f:
            pickle.dump(state, f)
        print(f"  [state saved — {len(accepted)} accepted, "
              f"{len(abandoned)} abandoned, "
              f"${cu_this_run/1e6*cu_rate_per_million:.3f} spent this run]")

# Final save
state.update({
    "results": results, "accepted": accepted,
    "abandoned": abandoned, "in_progress": in_progress,
    "total_cu": total_cu,
})
with open(STATE_PKL, "wb") as f:
    pickle.dump(state, f)

print(f"\nStage 2 run complete.")
print(f"  Accepted:       {len(accepted)}")
print(f"  Abandoned:      {len(abandoned)}")
print(f"  CUs this run:   {cu_this_run:,}")
print(f"  Cost this run:  ${cu_this_run/1e6*cu_rate_per_million:.3f}")


Candidates: 20 network + 279 stage1 = 50 total (capped at 50)

Starting Stage 2...
  Budget this run: 5,000,000 CUs (~$2.25)

[0xdfaa7532-i1] 31/31 batches (100.0%)[inc 1/6]  
[0xdfaa7532-i2] 31/31 batches (100.0%)
[0xdfaa7532-i3] 31/31 batches (100.0%)
[0xdfaa7532-i4] 31/31 batches (100.0%)
[0xdfaa7532-i5] 31/31 batches (100.0%)
[0xdfaa7532-i6] 31/31 batches (100.0%)
✗ abandoned edge=-0.4144%  n=113  LR=0.25
[0x974caa59-i1] 31/31 batches (100.0%)[inc 1/6]  
[0x974caa59-i2] 31/31 batches (100.0%)
[0x974caa59-i3] 31/31 batches (100.0%)
[0x974caa59-i4] 31/31 batches (100.0%)
[0x974caa59-i5] 31/31 batches (100.0%)
[0x974caa59-i6] 31/31 batches (100.0%)
✗ abandoned edge=-0.7415%  n=96  LR=0.33
[0x2af4feb9-i1] 31/31 batches (100.0%)[inc 1/6]  
[0x2af4feb9-i2] 31/31 batches (100.0%)
[0x2af4feb9-i3] 31/31 batches (100.0%)
[0x2af4feb9-i4] 31/31 batches (100.0%)
[0x2af4feb9-i5] 31/31 batches (100.0%)
[0x2af4feb9-i6] 31/31 batches (100.0%)
✓ ACCEPTED  edge=+0.8263%  n=0  LR=1.00
[0x881d4023-i1] 

## 8. Results analysis

In [9]:
# ================================================================
# Results analysis
# ================================================================

results_df = pd.DataFrame(list(results.values()))

if len(results_df) == 0:
    print("No results yet.")
else:
    accepted_df  = results_df[results_df["decision"]=="accept"].sort_values(
        "weighted_edge", ascending=False
    )
    abandoned_df = results_df[results_df["decision"]=="abandon"]

    print("=" * 60)
    print("STAGE 2 RESULTS SUMMARY")
    print("=" * 60)
    print(f"Total processed:       {len(results_df)}")
    print(f"Accepted (skilled):    {len(accepted_df)}")
    print(f"Abandoned:             {len(abandoned_df)}")
    print(f"In progress:           {len(results_df)-len(accepted_df)-len(abandoned_df)}")
    print()

    mean_inc = results_df["increments_used"].mean()
    tot_cu   = results_df["cu_used"].sum()
    print(f"Mean increments/wallet:  {mean_inc:.2f}")
    print(f"Total CU used:           {tot_cu:,}")
    print(f"Total cost:              ${tot_cu/1e6*cu_rate_per_million:.3f}")
    if len(results_df) > 0:
        print(f"Cost per wallet:         ${tot_cu/len(results_df)/1e6*cu_rate_per_million:.4f}")
    print()

    print("Early stopping effectiveness:")
    for inc in range(1, STAGE2_MAX_INCREMENTS + 1):
        n   = (results_df["increments_used"] == inc).sum()
        pct = n / len(results_df) * 100 if len(results_df) > 0 else 0
        print(f"  Stopped at increment {inc}: {n:3d} ({pct:.1f}%)")
    print()

    print("Weighted edge distribution:")
    print(results_df["weighted_edge"].describe().round(4))
    print()

    if len(accepted_df) > 0:
        print("Accepted wallets:")
        print(accepted_df[[
            "address","weighted_edge","n_trades","increments_used","likelihood_ratio"
        ]].round(4).to_string(index=False))
        accepted_df.to_csv(ACCEPTED_CSV, index=False)
        print(f"\nSaved to: {ACCEPTED_CSV}")
        print()
        print("Next steps:")
        print("  1. Run hourly balance fetch on accepted wallets")
        print("  2. Add to wallet pool and retrain walk-forward model")
        print("  3. Expect improved signal quality vs current random pool")
    else:
        print("No wallets accepted.")
        print("  Options:")
        print("  - Increase STAGE2_MAX_CANDIDATES to search more wallets")
        print("  - Relax STAGE2_SPRT_UPPER (currently {STAGE2_SPRT_UPPER})")
        print("  - Check Stage 1 candidate quality (are mean trade sizes reasonable?)")
        print("  - Run network expansion cell to get higher-quality leads")


STAGE 2 RESULTS SUMMARY
Total processed:       50
Accepted (skilled):    13
Abandoned:             37
In progress:           0

Mean increments/wallet:  6.00
Total CU used:           4,104,000
Total cost:              $1.847
Cost per wallet:         $0.0369

Early stopping effectiveness:
  Stopped at increment 1:   0 (0.0%)
  Stopped at increment 2:   0 (0.0%)
  Stopped at increment 3:   0 (0.0%)
  Stopped at increment 4:   0 (0.0%)
  Stopped at increment 5:   0 (0.0%)
  Stopped at increment 6:  50 (100.0%)

Weighted edge distribution:
count    50.0000
mean      0.0861
std       1.4629
min      -4.4187
25%      -0.3826
50%       0.0000
75%       0.2121
max       6.4776
Name: weighted_edge, dtype: float64

Accepted wallets:
                                   address  weighted_edge  n_trades  increments_used  likelihood_ratio
0x5b702b24b00a37b61ff9d71d91fed1a629edd1df         6.4776         0                6            1.0000
0x6ccfa5edac4a4dc8e18f2cde6008aff46002c924         4.5815    

In [10]:
# Check which accepted wallets have real trade counts
accepted_df = pd.read_csv(ACCEPTED_CSV)
print("Accepted wallets — quality tiers:")
print()
print("HIGH quality (n_trades >= 10, edge > 0.5%):")
high = accepted_df[(accepted_df['n_trades'] >= 10) & 
                   (accepted_df['weighted_edge'] > 0.5)]
print(high[['address','weighted_edge','n_trades','likelihood_ratio']].to_string(index=False))
print()
print("MEDIUM quality (n_trades >= 3):")
med = accepted_df[(accepted_df['n_trades'] >= 3) & 
                  (accepted_df['n_trades'] < 10)]
print(med[['address','weighted_edge','n_trades','likelihood_ratio']].to_string(index=False))
print()
print("LOW quality (n_trades < 3 — likely data gap):")
low = accepted_df[accepted_df['n_trades'] < 3]
print(low[['address','weighted_edge','n_trades']].to_string(index=False))

Accepted wallets — quality tiers:

HIGH quality (n_trades >= 10, edge > 0.5%):
                                   address  weighted_edge  n_trades  likelihood_ratio
0x86a067030a9668c13ff2a8c4d5415afc776d4c63       1.521406        18          0.969440
0xb685760ebd368a891f27ae547391f4e2a289895b       0.923972        97          0.394073

MEDIUM quality (n_trades >= 3):
                                   address  weighted_edge  n_trades  likelihood_ratio
0x88ac52fb7a432fe1b87ba04f3782a91bd981cc77       1.070335         3               1.0
0x2fa0f4df54d62faab9e90a97c792b3e026fb50b7       1.058095         4               1.0
0x44894aeee56c2dd589c1d5c8cb04b87576967f97       0.214716         6               1.0

LOW quality (n_trades < 3 — likely data gap):
                                   address  weighted_edge  n_trades
0x5b702b24b00a37b61ff9d71d91fed1a629edd1df       6.477620         0
0x6ccfa5edac4a4dc8e18f2cde6008aff46002c924       4.581516         0
0x6131b5fae19ea4f9d964eac0408e4408b

## 9. Cost summary & run plan

In [11]:
# ================================================================
# Full cost summary and run plan
# ================================================================

print("=" * 65)
print("WALLET SEARCH v2 — COST SUMMARY AND RUN PLAN")
print("=" * 65)
print()
print("LEVERS (cell 1):")
print(f"  STAGE1_MAX_LOG_CALLS  = {STAGE1_MAX_LOG_CALLS:4d}  → ${STAGE1_MAX_LOG_CALLS*cu_per_log_call/1e6*cu_rate_per_million:.3f}")
print(f"  STAGE2_MAX_CANDIDATES = {STAGE2_MAX_CANDIDATES:4d}  → ${STAGE2_MAX_CANDIDATES*2*STAGE2_SCORING_DAYS*24*cu_per_balance/1e6*cu_rate_per_million:.2f} (expected)")
print(f"  MAX_CU_THIS_RUN       = {MAX_CU_THIS_RUN:,}  → ${MAX_CU_THIS_RUN/1e6*cu_rate_per_million:.2f} cap")
print()
print("SUGGESTED RUNS:")
print()
r1 = 100*cu_per_log_call/1e6*cu_rate_per_million
r2 = 50*2*STAGE2_SCORING_DAYS*24*cu_per_balance/1e6*cu_rate_per_million
r3 = 500*2*STAGE2_SCORING_DAYS*24*cu_per_balance/1e6*cu_rate_per_million
print(f"  Run 1 — Stage 1 + network only:        ~${r1:.3f}")
print(f"    Set STAGE1_MAX_LOG_CALLS=100, skip Stage 2")
print(f"    Review candidates before spending on Stage 2")
print()
print(f"  Run 2 — Stage 2 small batch:           ~${r2:.2f}")
print(f"    Set STAGE2_MAX_CANDIDATES=50, MAX_CU_THIS_RUN=1_000_000")
print(f"    Validate SPRT and early stopping are working")
print()
print(f"  Run 3 — Stage 2 full (overnight):      ~${r3:.2f}")
print(f"    Set STAGE2_MAX_CANDIDATES=500, MAX_CU_THIS_RUN=10_000_000")
print(f"    Restart-safe — re-run cell to continue if interrupted")
print()
print(f"  Run 4 — Network expansion:             ~$0.10")
print(f"    After any wallets are accepted")
print(f"    High-quality leads from confirmed skilled wallet graph")
print()
print(f"  Total expected: ~${r1+r2+r3+0.1:.2f}")
print(f"  vs $117 spent on random sampling to date")


WALLET SEARCH v2 — COST SUMMARY AND RUN PLAN

LEVERS (cell 1):
  STAGE1_MAX_LOG_CALLS  =  100  → $0.003
  STAGE2_MAX_CANDIDATES =   50  → $0.62 (expected)
  MAX_CU_THIS_RUN       = 5,000,000  → $2.25 cap

SUGGESTED RUNS:

  Run 1 — Stage 1 + network only:        ~$0.003
    Set STAGE1_MAX_LOG_CALLS=100, skip Stage 2
    Review candidates before spending on Stage 2

  Run 2 — Stage 2 small batch:           ~$0.62
    Set STAGE2_MAX_CANDIDATES=50, MAX_CU_THIS_RUN=1_000_000
    Validate SPRT and early stopping are working

  Run 3 — Stage 2 full (overnight):      ~$6.16
    Set STAGE2_MAX_CANDIDATES=500, MAX_CU_THIS_RUN=10_000_000
    Restart-safe — re-run cell to continue if interrupted

  Run 4 — Network expansion:             ~$0.10
    After any wallets are accepted
    High-quality leads from confirmed skilled wallet graph

  Total expected: ~$6.87
  vs $117 spent on random sampling to date


In [12]:
# Check actual CU usage from the 50-wallet run
with open(SEARCH_DIR / 'search_state.pkl', 'rb') as f:
    old_state = pickle.load(f)

results_df = pd.DataFrame(list(old_state['results'].values()))
print(f"Wallets processed: {len(results_df)}")
print(f"CU used (tracked): {old_state['total_cu']:,}")
print(f"CU per wallet:     {old_state['total_cu']/len(results_df):,.0f}")
print()
print(f"Actual Alchemy reported: ~16,000 CUs")
print(f"Actual cost per wallet:  ${16000/50/1e6*0.45:.6f}")
print(f"Projected 10k cost:      ${16000/50*10000/1e6*0.45:.2f}")

Wallets processed: 50
CU used (tracked): 4,109,700
CU per wallet:     82,194

Actual Alchemy reported: ~16,000 CUs
Actual cost per wallet:  $0.000144
Projected 10k cost:      $1.44
